## Cell 1 — Import the tools

In [10]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
print("Libraries loaded!")

Libraries loaded!


## Cell 2 — Load the data and look at it

In [11]:
# The raw file has NO column names — just numbers. We name the 26 columns ourselves.
col_names = ['unit_number', 'time_in_cycles',
             'op_setting_1', 'op_setting_2', 'op_setting_3'] \
            + [f'sensor_{i:02d}' for i in range(1, 22)]   # sensor_01 ... sensor_21

# Read the file. Values are separated by spaces, and there's no header row.
df = pd.read_csv('train_FD001.txt', sep=r'\s+', header=None, names=col_names)

# Show the size and the first 5 rows so we can see what we're working with.
print("Shape (rows, columns):", df.shape)
df.head()

Shape (rows, columns): (20631, 26)


,unit_number,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


## Cell 3 — Build the RUL target (the "answer" to learn)

In [12]:
# Each engine's final cycle = its failure point
last_cycle = df.groupby('unit_number')['time_in_cycles'].max()
df['RUL'] = df['unit_number'].map(last_cycle) - df['time_in_cycles']

# Cap at 125
df['RUL'] = df['RUL'].clip(upper=125)

print("RUL target built. Example — engine 1, last 5 rows:")
df[df['unit_number'] == 1][['time_in_cycles', 'RUL']].tail()

RUL target built. Example — engine 1, last 5 rows:


,time_in_cycles,RUL
187,188,4
188,189,3
189,190,2
190,191,1
191,192,0


## Cell 4 — Choose the inputs and the answer

In [13]:
feature_cols = ['op_setting_1', 'op_setting_2', 'op_setting_3'] \
               + [f'sensor_{i:02d}' for i in range(1, 22)]

X = df[feature_cols]   # inputs (24 columns)
y = df['RUL']          # the answer

print("Inputs:", X.shape, "| Answer:", y.shape)

Inputs: (20631, 24) | Answer: (20631,)


## Cell 5 — Split by engine (leak-free)

In [14]:
engine_ids = df['unit_number'].unique()
train_engines = engine_ids[:80]     # first 80 engines to learn from
test_engines  = engine_ids[80:]     # last 20 kept for honest testing

train_mask = df['unit_number'].isin(train_engines)
X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[~train_mask], y[~train_mask]

print(f"Train rows: {len(X_train):,} | Test rows: {len(X_test):,}")

Train rows: 16,138 | Test rows: 4,493


## Cell 6 — Build and train the model

In [15]:
model = Pipeline([
    ('scaler', MinMaxScaler()),                              # rescale inputs fairly
    ('rf', RandomForestRegressor(n_estimators=200, random_state=42))  # the model
])

model.fit(X_train, y_train)   # <-- training happens here
print("Model trained!")

Model trained!


## Cell 7 — Check how good it is

In [16]:
preds = model.predict(X_test)

mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2   = r2_score(y_test, preds)

print(f"MAE:  {mae:.2f} cycles")
print(f"RMSE: {rmse:.2f} cycles")
print(f"R2:   {r2:.3f}")

MAE:  14.46 cycles
RMSE: 19.18 cycles
R2:   0.784


## Cell 8 — Turn a prediction into a decision + test one engin

In [17]:
def rul_to_risk(rul):
    if rul < 30:   return "High"      # inspect now
    elif rul < 75: return "Medium"    # schedule soon
    else:          return "Low"       # routine monitoring

# Try it on the first test engine's most recent reading
example = X_test.iloc[[0]]
pred_rul = model.predict(example)[0]
print(f"Predicted RUL: {pred_rul:.1f} cycles  ->  Risk: {rul_to_risk(pred_rul)}")

Predicted RUL: 110.3 cycles  ->  Risk: Low


## Cell 9 — Save the model (the bridge to the backend)

In [19]:
import joblib, json, sklearn
from datetime import datetime
from google.colab import files

# Save the model
joblib.dump(model, 'model.pkl')

# Save the metadata (what the backend reads to stay in sync)
metadata = {
    "model_type": "RandomForestRegressor",
    "dataset": "NASA C-MAPSS FD001",
    "target": "RUL (clipped at 125)",
    "rul_clip": 125,
    "n_features": len(feature_cols),
    "feature_names": feature_cols,
    "metrics": {"MAE": round(mae, 2), "RMSE": round(rmse, 2), "R2": round(r2, 3)},
    "risk_thresholds": {"high": 30, "medium": 75},
    "sklearn_version": sklearn.__version__,
    "trained_at": datetime.now().isoformat(timespec="seconds"),
}
json.dump(metadata, open('model_metadata.json', 'w'), indent=2)

print("Saved model.pkl and model_metadata.json")
files.download('model.pkl')              # downloads to your computer
files.download('model_metadata.json')

Saved model.pkl and model_metadata.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>